In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg2:///olist_db")

out = Path("tableau_exports")
out.mkdir(exist_ok=True)

# -----------------------------
# 1. Pull source tables/views
# -----------------------------
orders = pd.read_sql("SELECT * FROM analytics.v_orders_enriched", engine)
customers = pd.read_sql("SELECT * FROM analytics.v_customer_summary", engine)
category_perf = pd.read_sql("SELECT * FROM analytics.v_category_performance", engine)
seller_perf = pd.read_sql("SELECT * FROM analytics.v_seller_performance", engine)
state_perf = pd.read_sql("SELECT * FROM analytics.v_state_performance", engine)

# -----------------------------
# 2. Date conversions
# -----------------------------
orders["purchase_month_start"] = pd.to_datetime(orders["purchase_month_start"], errors="coerce")
orders["purchase_date"] = pd.to_datetime(orders["purchase_date"], errors="coerce")

category_perf["purchase_month_start"] = pd.to_datetime(category_perf["purchase_month_start"], errors="coerce")
seller_perf["purchase_month_start"] = pd.to_datetime(seller_perf["purchase_month_start"], errors="coerce")
state_perf["purchase_month_start"] = pd.to_datetime(state_perf["purchase_month_start"], errors="coerce")

# -----------------------------
# 3. Delivered-only clean slice
# Exclude incomplete tail months
# -----------------------------
delivered = orders[
    (orders["order_status"] == "delivered") &
    (orders["purchase_month_start"] < "2018-09-01")
].copy()

delivered["order_value_proxy"] = delivered["order_value_proxy"].fillna(0)
delivered["product_revenue"] = delivered["product_revenue"].fillna(0)
delivered["freight_revenue"] = delivered["freight_revenue"].fillna(0)
delivered["avg_review_score"] = pd.to_numeric(delivered["avg_review_score"], errors="coerce")
delivered["delivery_days"] = pd.to_numeric(delivered["delivery_days"], errors="coerce")
delivered["late_delivery_flag"] = pd.to_numeric(delivered["late_delivery_flag"], errors="coerce").fillna(0).astype(int)

# -----------------------------
# 4. Executive monthly KPIs
# -----------------------------
executive_monthly_kpis = (
    delivered.groupby("purchase_month_start", as_index=False)
    .agg(
        delivered_orders=("order_id", "nunique"),
        delivered_gpv=("order_value_proxy", "sum"),
        avg_review_score=("avg_review_score", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        late_delivery_rate=("late_delivery_flag", "mean")
    )
    .sort_values("purchase_month_start")
)

executive_monthly_kpis.to_csv(out / "executive_monthly_kpis.csv", index=False)

# -----------------------------
# 5. Customer summary
# -----------------------------
customer_summary = customers.copy()
customer_summary.to_csv(out / "customer_summary.csv", index=False)

# -----------------------------
# 6. State summary
# -----------------------------
state_summary = (
    state_perf.groupby("customer_state", as_index=False)
    .agg(
        delivered_orders=("delivered_orders", "sum"),
        delivered_gpv=("delivered_gpv", "sum"),
        avg_review_score=("avg_review_score", "mean"),
        avg_delivery_days=("avg_delivery_days", "mean"),
        late_delivery_rate=("late_delivery_rate", "mean")
    )
    .sort_values("delivered_gpv", ascending=False)
)

state_summary.to_csv(out / "state_summary.csv", index=False)

# -----------------------------
# 7. Category summary
# -----------------------------
category_summary = (
    category_perf.groupby("product_category_en", as_index=False)
    .agg(
        delivered_orders=("delivered_orders", "sum"),
        product_revenue=("product_revenue", "sum"),
        freight_revenue=("freight_revenue", "sum"),
        avg_review_score=("avg_review_score", "mean"),
        avg_delivery_days=("avg_delivery_days", "mean"),
        late_delivery_rate=("late_delivery_rate", "mean")
    )
    .sort_values("product_revenue", ascending=False)
)

category_summary.to_csv(out / "category_summary.csv", index=False)

# -----------------------------
# 8. Seller summary
# -----------------------------
seller_summary = (
    seller_perf.groupby(["seller_id", "seller_state"], as_index=False)
    .agg(
        delivered_orders=("delivered_orders", "sum"),
        product_revenue=("product_revenue", "sum"),
        avg_review_score=("avg_review_score", "mean"),
        avg_delivery_days=("avg_delivery_days", "mean"),
        late_delivery_rate=("late_delivery_rate", "mean")
    )
    .sort_values("product_revenue", ascending=False)
)

seller_summary.to_csv(out / "seller_summary.csv", index=False)

# -----------------------------
# 9. Delivery experience summary
# -----------------------------
delivery_experience_summary = (
    delivered.assign(
        delivery_bucket=np.where(delivered["late_delivery_flag"] == 1, "Late", "On Time / Early")
    )
    .groupby("delivery_bucket", as_index=False)
    .agg(
        orders=("order_id", "count"),
        avg_review_score=("avg_review_score", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_order_value=("order_value_proxy", "mean")
    )
)

delivery_experience_summary.to_csv(out / "delivery_experience_summary.csv", index=False)

print("Export complete. Files created:")
for f in sorted(out.glob("*.csv")):
    print("-", f.name)

Export complete. Files created:
- category_summary.csv
- customer_summary.csv
- delivery_experience_summary.csv
- executive_monthly_kpis.csv
- seller_summary.csv
- state_summary.csv


In [3]:
executive_kpi_snapshot = pd.DataFrame([{
    "total_orders": orders["order_id"].nunique(),
    "delivered_orders": delivered["order_id"].nunique(),
    "delivered_rate_pct": round(delivered["order_id"].nunique() / orders["order_id"].nunique() * 100, 2),
    "delivered_gpv": round(delivered["order_value_proxy"].sum(), 2),
    "avg_order_value": round(delivered["order_value_proxy"].mean(), 2),
    "avg_review_score": round(delivered["avg_review_score"].mean(), 2),
    "avg_delivery_days": round(delivered["delivery_days"].mean(), 2),
    "late_delivery_rate_pct": round(delivered["late_delivery_flag"].mean() * 100, 2)
}])

executive_kpi_snapshot.to_csv(out / "executive_kpi_snapshot.csv", index=False)

In [4]:
list(sorted(Path("tableau_exports").glob("*.csv")))

[PosixPath('tableau_exports/category_summary.csv'),
 PosixPath('tableau_exports/customer_summary.csv'),
 PosixPath('tableau_exports/delivery_experience_summary.csv'),
 PosixPath('tableau_exports/executive_kpi_snapshot.csv'),
 PosixPath('tableau_exports/executive_monthly_kpis.csv'),
 PosixPath('tableau_exports/seller_summary.csv'),
 PosixPath('tableau_exports/state_summary.csv')]